# Nonlinear Probabilistic Reconciliation of Swiss immigration rate via conditioning

## Introduction

This notebook demonstrates the application of nonlinear probabilistic reconciliation via conditioning to the Swiss immigration rate that we publish in this package. The data are obtained from the *Swiss Federal Statistical Office (FSO)* of Switzerland, available through the [FSO PX-Web API](https://www.pxweb.bfs.admin.ch/api/v1/en/px-x-0102020000_104/px-x-0102020000_104.px). Switzerland is subdivided into 26 cantons.

For each Canton and for whole Switzerland, we have extracted annual counts of immigration flow, population (as of 1st January) from years 1981 to 2024. These correspond to the demographic components required to construct the immigration-to-population ratio.

The immigration structure is illustrated in the figure below for two cantons among the 26 cantons in Switzerland. Here, $P$, $I$, and $R$ denote the foreign population, the number of immigrants, and the immigration rate, respectively. The superscripts $AG$, $ZH$, and $CH$ refer to the Canton of Aargau, the Canton of Zürich, and Switzerland as a whole.

Thus, there are :
-   $54$ free time series corresponding to the Cantonal level counts of $I$ and $P$ (including No indications), and
-   $29$ constrained time series comprising the Cantonal and Swiss immigration ratios along with the total counts of $I$ and $P$ for the whole of Switzerland for this dataset.

![CH.png](../pictures/CH.png)

In [1]:
#imports

import warnings
warnings.filterwarnings("ignore")

import time
import pickle
import numpy as np
from pathlib import Path

from nonlinear.conditioning import reconc_nl_is, reconc_nl_ukf
from utils.stats import _schafer_strimmer_cov

In this notebook we will reconcile the base forecasts for all these series, ensuring that the forecasts follow the nonlinear and linear constraints. The base forecasts are already available in the package fitted using Auto ARIMA models.

In [8]:
# ------------------------------------------------------------------
# Load base forecasts and test data
# ------------------------------------------------------------------

forecast_dir = Path("../data")

def load_pickle(filename):
    path = forecast_dir / filename
    with path.open("rb") as f:
        return pickle.load(f)

base = load_pickle("fc_imm_cit_autoarima2_30.pkl")
base_2 = load_pickle("fc_imm_cit_autoarima_30.pkl")
test_data = load_pickle("test_autoarima_30.pkl")
test_data_2 = load_pickle("test_autoarima2_30.pkl")

print("base keys:", base.keys())
print("base_2 keys:", base_2.keys())

base keys: dict_keys(['immigration', 'population', 'citizenship', 'immigration_ratio', 'citizenship_ratio'])
base_2 keys: dict_keys(['cantons_immigration', 'cantons_population', 'cantons_citizenship', 'Switzerland_immigration', 'Switzerland_population', 'Switzerland_citizenship', 'Switzerland_immigration_ratio', 'Switzerland_citizenship_ratio'])


In [3]:
def pick_block(block_key, base=base):
        S = base[block_key]["samples"]
        R = base[block_key]["residuals"]
        return S, R

pop_free, pop_free_res = pick_block("population", base)
imm_free, imm_free_res = pick_block("immigration", base)
cit_free, cit_free_res = pick_block("citizenship", base)
U, T, M = pop_free.shape
ref_uids = base["population"]["uids"]
no_ind_pos = np.where(np.array(ref_uids) == 'No indication')[0].tolist()
pop_total, pop_total_res = pick_block("Switzerland_population", base_2)
imm_total, imm_total_res = pick_block("Switzerland_immigration", base_2)
imm_ratio, imm_ratio_res = pick_block("immigration_ratio", base)
rat_uids = base['immigration_ratio']['uids']
ch_uid_pos = np.where(np.array(rat_uids) == 'Switzerland')[0]
imm_ratio_mid, imm_ratio_mid_res = np.delete(imm_ratio, ch_uid_pos, axis=0), np.delete(imm_ratio_res, ch_uid_pos, axis=0)
imm_ratio_top, imm_ratio_top_res = imm_ratio[ch_uid_pos,:,:], imm_ratio_res[ch_uid_pos,:,:]

In [4]:
def f_constrained_from_free(Bns, ref_id):
    """
    Vectorized map f: free -> constrained for BUIS.
    free Bns: (N, 2U) with order [imm(U), pop(U)].
    Returns constrained: (U+1, N) with order [mid_ratios(U), top_ratio(1)].
    where mid_k = imm_k / pop_k, and top = sum_k w_k * mid_k.
    """
    U = Bns.shape[1] // 2
    imm = Bns[:, :U]
    pop = Bns[:, U:]
    mid_total = np.concatenate([np.sum(imm, axis=1, keepdims=True), np.sum(pop, axis=1, keepdims=True)], axis=1)
    no_ind_pos = np.where(np.array(ref_id) == 'No indication')[0].tolist()
    mid_ratio = np.delete(imm, no_ind_pos, axis=1) / (np.delete(pop, no_ind_pos, axis=1) + 1e-8)
    top = np.sum(imm, axis=1, keepdims=True) / np.sum(pop, axis=1, keepdims=True)
    constrained = np.concatenate([top, mid_ratio, mid_total], axis=1)
    return constrained.T

def f_constrained_from_free_single(z, no_ind_pos):
    U = z.shape[0] // 2
    imm = z[:U]
    pop = z[U:]
    mid_total = np.concatenate([np.atleast_1d(np.sum(imm)), np.atleast_1d(np.sum(pop))])
    mid_ratio = np.delete(imm, no_ind_pos) / (np.delete(pop, no_ind_pos) + 1e-8)
    top = np.atleast_1d(np.sum(imm) / np.sum(pop))
    return np.concatenate([top, mid_ratio, mid_total])


Below we demonstrate the application of the nonlinear UKF-based reconciliation via conditioning, and the IS-based reconciliation via conditioning. 

The UKF-based method is applied by using the function  `reconc_nl_ukf`  that takes the following as input:
-  `free base forecasts`  : a list of $n_b$ dictionaries, each containing "samples" and "residuals" for the free series
-  `in_type`  : a list of length $n_b$ indicating the type of each free series (e.g., "samples" or "distr")
-  `distr`  : a list of length $n_b$ containing the distribution type for each free series (e.g., "gaussian"); can be empty if in_type is "samples"
-  `f_u`  : FTC function 
-  `constrained base forecasts`  : array of shape (n_u) containing the mean vector of base forecasts for the constrained series
-  `R`  : array of shape (n_u, n_u) containing the covariance matrix of the constrained series (estimated from residuals)
-  `num_samples`  : number of reconciled samples to generate
-  `seed`  : random seed for reproducibility

In [5]:
t0_ukf_imm = time.perf_counter()
nlukf_imm = {}

for t in range(T):
    u_obs = np.mean(np.vstack([imm_ratio_top[:, t, :], imm_ratio_mid[:, t, :],
                                   imm_total[:,t,:], pop_total[:,t,:]]), axis=1)
    R = _schafer_strimmer_cov((np.vstack([imm_ratio_top_res[:, t, :],
                                              imm_ratio_mid_res[:, t, :],
                                              imm_total_res[:,t,:],
                                              pop_total_res[:,t,:]])).T)['shrink_cov']

    free_list = []
    for k in range(U):  # immigration totals
        free_list.append({"samples": imm_free[k, t, :], "residuals": imm_free_res[k, t, :]})
    for k in range(U):  # population totals
        free_list.append({"samples": pop_free[k, t, :], "residuals": pop_free_res[k, t, :]})

    def f_single(z):
        return f_constrained_from_free_single(z, no_ind_pos)

    out = reconc_nl_ukf(
        free_base_forecasts=free_list,
        in_type=["samples"] * (2 * U),
        distr=[] * (2 * U),
        f_u=f_single,
        constrained_base_forecasts=u_obs,
        R=R,
        num_samples=M,
        seed=42,
    )
    Brec = out["free_reconciled_samples"]
    Urec = f_constrained_from_free(Brec.T, ref_id=ref_uids)
    nlukf_imm[t] = np.vstack([Urec, Brec])

nlukf_imm = np.stack([nlukf_imm[t] for t in range(T)], axis=1)
print("Time taken to reconcile:", time.perf_counter() - t0_ukf_imm,"seconds")
print("UKF reconciled sample shape:", nlukf_imm.shape)

Time taken to reconcile: 1.693131241001538 seconds
UKF reconciled sample shape: (83, 30, 1000)


The IS-based method is applied by using the function  `reconc_nl_is`  that takes the following as input:
-  `joint_mean`  : array of shape (n_u + n_b) containing the mean vector of the joint base forecasts for both constrained and free series
-  `joint_cov`  : array of shape (n_u + n_b, n_u + n_b) containing the covariance matrix of the joint base forecasts for both constrained and free series (estimated from residuals)
-  `n_free`  : number of free series (to identify the split between constrained and free in the joint vector)
-  `f_u`  : vectorized FTC function
- `num_samples`  : number of reconciled samples to generate
- `seed`  : random seed for reproducibility

In [7]:
def f_is(B):
        """
        B: (M, 2U) = [imm(U), pop(U)]
        Returns:
            constrained: (1 + (U-1) + 2, M)
                 = [top_ratio(1), mid_ratios(U-1), swiss_totals(2)]
        """
        P = B.shape[1] // 2
        imm = B[:, :P]
        pop = B[:, P:]

        mid_total = np.concatenate(
            [
                np.sum(imm, axis=1, keepdims=True),
                np.sum(pop, axis=1, keepdims=True),
            ],
            axis=1,
        )  # (M, 2)

        mid_ratio = np.delete(imm, no_ind_pos, axis=1) / (
                np.delete(pop, no_ind_pos, axis=1) + 1e-8
        )  # (M, U-1)

        top = np.sum(imm, axis=1, keepdims=True) / (
                np.sum(pop, axis=1, keepdims=True) + 1e-8
        )  # (M, 1)

        constrained = np.concatenate([top, mid_ratio, mid_total], axis=1)  # (M, 1+(U-1)+2)
        return constrained.T  # (n_constrained, M)

t0_is_imm = time.perf_counter()
nl_is_imm = {}

for t in range(T):
        # ----------------------------------------
        # free base samples: [imm_frees, pop_frees]
        # shape: (2U, M)
        # ----------------------------------------
        fc_free_arr = np.vstack([
            imm_free[:, t, :],
            pop_free[:, t, :]
        ])

        # ----------------------------------------
        # constrained base samples: [top_ratio, mid_ratios, swiss_totals]
        # shape: (1 + (U-1) + 2, M)
        # ----------------------------------------
        fc_constrained_arr = np.vstack([
            imm_ratio_top[:, t, :],
            imm_ratio_mid[:, t, :],
            imm_total[:, t, :],
            pop_total[:, t, :]
        ])

        n_constrained = fc_constrained_arr.shape[0]
        n_free = fc_free_arr.shape[0]

        # Full joint base vector [U, B]
        joint_base = np.vstack([fc_constrained_arr, fc_free_arr])  # (n_constrained + n_free, M)

        # Joint Gaussian approximation from residuals of the same full structure
        joint_resid = np.concatenate([
            imm_ratio_top_res[:, t, :].T,
            imm_ratio_mid_res[:, t, :].T,
            imm_total_res[:, t, :].T,
            pop_total_res[:, t, :].T,
            imm_free_res[:, t, :].T,
            pop_free_res[:, t, :].T,
        ], axis=1)  # (window_or_res_samples, n_constrained+n_free)

        joint_cov = _schafer_strimmer_cov(joint_resid)["shrink_cov"]
        joint_mean = np.mean(joint_base, axis=1)

        buis_res = reconc_nl_is(
            joint_mean=joint_mean,
            joint_cov=joint_cov,
            n_free=n_free,
            f_u=f_is,
            num_samples=M,
            seed=42,
        )

        nl_is_imm[t] = buis_res["reconciled_samples"]

is_imm = np.stack([nl_is_imm[t] for t in range(T)], axis=1)
print("Time taken to reconcile:", time.perf_counter() - t0_is_imm,"seconds")
print("NL-IS reconciled sample shape:", is_imm.shape)

Time taken to reconcile: 1.2289514650001365 seconds
NL-IS reconciled sample shape: (83, 30, 1000)


Thus, we can see that both the conditioning-based methods are fast, and produce reconciled forecasts with the same shape as the base forecasts.

## Evaluation of probabilistic forecasts via CRPS

In [13]:
test_imm_ratio = test_data_2['immigration_ratio']['y_true']
test_imm_total = test_data['Switzerland_immigration']['y_true'].reshape(1,30)
test_pop_total = test_data['Switzerland_population']['y_true'].reshape(1,30)
test_imm = test_data_2['immigration']['y_true']
test_pop = test_data_2['population']['y_true']

test_imm_data = np.vstack([test_imm_ratio[ch_uid_pos,:],
                                    np.delete(test_imm_ratio, ch_uid_pos, axis=0),
                                    test_imm_total,
                                    test_pop_total,
                                    test_imm,
                                    test_pop])

base_imm = np.vstack([imm_ratio_top,
                              imm_ratio_mid,
                              imm_total,
                              pop_total,
                              imm_free,
                              pop_free])


methods_imm = {
        "Base": base_imm,
        "UKF": nlukf_imm,
        "IS": is_imm
    }



We compute a score based on the geometric mean of the relative CRPS across time series:
\begin{equation*}
    \mathrm{RelCRPS}_{\text{method}}
    =
    \left(
    \prod_{j=1}^{n}
    \frac{\mathrm{CRPS}_{j,\text{method}}}
         {\mathrm{CRPS}_{j,\text{base}}}
    \right)^{1/n},
\end{equation*}
where
\begin{equation*}
    \mathrm{CRPS}_{j,\text{method}}
    =
    \frac{1}{T}
    \sum_{t=1}^{T}
    \mathrm{CRPS}\!\left(
    \hat{F}^{\,\text{method}}_{j,t},\, y_{j,t}
    \right),
\end{equation*}
and analogously for $\mathrm{CRPS}_{j,\text{base}}$.
Here, $T$ denotes the number of rolling windows (or evaluation time steps), $\hat{F}^{\,\text{method}}_{j,t}$ is the predictive distribution produced by the considered method for series $j$ at time step $t$, and $y_{j,t}$ is the corresponding ground truth.

In [18]:
def compute_crps(y_section, y_hat_section, q_min=None):
    start_from = 0 if q_min is None else int(y_hat_section.shape[1] * q_min)
    y_hat_sorted = np.sort(y_hat_section[:, start_from:], axis=1)
    m = y_hat_sorted.shape[1]
    return (2 / m) * np.nanmean((y_hat_sorted - y_section[:, None]) *
                                (m * (y_section[:, None] < y_hat_sorted) - np.arange(1, m + 1) + 0.5),
                                axis=1)

def crps_relative_geomean_over_series(
            forecast_methods: dict,
            gt: np.ndarray,
            baseline_name: str = "Base",
            eps: float = 1e-12,
    ):
        """
        For each series j:
          1) compute avg CRPS over time: CRPS_method[j] = mean_t CRPS(y_true[j,t], y_samples[j,t,:])
          2) compute ratio per method: r_method[j] = CRPS_method[j] / CRPS_base[j]
          3) aggregate over j with geometric mean: GM = exp(mean_j log(r_method[j]))

        Parameters
        ----------
        forecast_methods : dict[name -> y_samples]
            Each y_samples has shape (R, T, M).
        gt : array (R, T)
            Ground truth.
        baseline_name : str
            Key in forecast_methods to use as baseline; if missing, first entry is used.
        eps : float
            Numerical floor to avoid divide-by-zero / log(0).

        Returns
        -------
        per_series_avg : dict[name -> array (R,)]
            Average CRPS per series (avg over time).
        per_series_ratio : dict[name -> array (R,)]
            Per-series ratio to baseline.
        gm_ratio : dict[name -> float]
            Geometric mean of per-series ratios (ignoring NaNs).
        """
        # pick baseline
        if baseline_name in forecast_methods:
            base_key = baseline_name
        else:
            base_key = next(iter(forecast_methods.keys()))

        R, T = gt.shape
        per_series_avg = {}

        # --- avg CRPS per series for each method
        for name, y_hat in forecast_methods.items():
            if y_hat.shape[:2] != (R, T):
                raise ValueError(f"{name}: expected shape (R,T,M) with (R,T)=({R},{T}), got {y_hat.shape}")

            avg_j = np.full(R, np.nan, dtype=float)
            for j in range(R):
                crps_t = []
                for t in range(T):
                    # compute_crps expects y_section shape (n_series,) and y_hat_section (n_series, M)
                    c = compute_crps(gt[j:j + 1, t], y_hat[j:j + 1, t, :])[0]
                    crps_t.append(c)
                avg_j[j] = np.nanmean(crps_t)
            per_series_avg[name] = avg_j

        # --- ratios to baseline, per series
        base_avg = per_series_avg[base_key]
        base_safe = np.where(np.isfinite(base_avg) & (base_avg > 0), base_avg, np.nan)

        per_series_ratio = {}
        gm_ratio = {}

        for name, avg_j in per_series_avg.items():
            ratio = avg_j / base_safe
            # avoid 0 or negative (shouldn't happen for CRPS, but protect logs)
            ratio = np.where(np.isfinite(ratio), np.maximum(ratio, eps), np.nan)
            per_series_ratio[name] = ratio

            valid = np.isfinite(ratio)
            if np.any(valid):
                gm_ratio[name] = float(np.exp(np.nanmean(np.log(ratio[valid]))))
            else:
                gm_ratio[name] = np.nan

        return per_series_avg, per_series_ratio, gm_ratio

def crps_gm_table(
            forecast_methods: dict,
            gt: np.ndarray,
            baseline_name: str = "Base",
    ):
        """
        Convenience printer: shows GM over series of (CRPS_method_j / CRPS_base_j),
        plus the baseline GM (=1 by construction if baseline exists).
        """
        per_series_avg, per_series_ratio, gm_ratio = crps_relative_geomean_over_series(
            forecast_methods, gt, baseline_name=baseline_name
        )

        # print
        base_key = baseline_name if baseline_name in forecast_methods else next(iter(forecast_methods.keys()))
        print(f"Baseline: {base_key}")
        print(f"{'Method':<12s} | {'GM(CRPS/BASE)':>14s}")
        print("-" * 30)
        for name in forecast_methods.keys():
            val = gm_ratio.get(name, np.nan)
            if np.isfinite(val):
                print(f"{name:<12s} | {val:>14.3f}x")
            else:
                print(f"{name:<12s} | {'nan':>14s}")

        return gm_ratio

crps_gm_table(methods_imm, test_imm_data, baseline_name="Base")

Baseline: Base
Method       |  GM(CRPS/BASE)
------------------------------
Base         |          1.000x
UKF          |          0.906x
IS           |          0.913x


{'Base': 1.0, 'UKF': 0.9062719319444571, 'IS': 0.9134663720985812}

Hence, the UKF approach is slightly more accurate than the IS-based approach of sampling from the reconciled distribution obtained via conditioning. Both the approaches however, are significantly better than the base forecasts that do not satisfy the constraints. This illustrates the benefits of probabilistic reconciliation via conditioning, even in a nonlinear setting.